In [1]:
import sys
# TO CHANGE
BASEDIR = "../../.."
sys.path.insert(0, BASEDIR)

In [2]:
from pprint import pprint

In [3]:
from src import PersonalAI, PersonalAIConfig
from src.kg_model import KnowledgeGraphModelConfig
from src.db_drivers.vector_driver.embedders import EmbedderModelConfig

from src.pipelines.qa import QAPipelineConfig
from src.pipelines.qa.kg_reasoning import KnowledgeGraphReasonerConfig
from src.pipelines.qa.query_preprocessing import QueryPreprocessorConfig
from src.pipelines.qa.kg_reasoning.weak_reasoner import WeakKGReasonerConfig
from src.pipelines.qa.kg_reasoning.medium_reasoner import MediumKGReasonerConfig

/home/dzigen/Desktop/Projects/PersonalAI/.pai_venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
kg_config = KnowledgeGraphModelConfig(
    nodestree_config=None # Модель дерева вершин строиться не будет
)
EMBEDDER_MODEL_PATH = '../../../models/intfloat/multilingual-e5-small' # PATH TO APPROPRIATE EMBEDDER-MODEL
kg_config.embedders_configs['m-e5-small'] = EmbedderModelConfig(model_name_or_path=EMBEDDER_MODEL_PATH)

In [24]:
pai_config = PersonalAIConfig(
    kg_model_config=kg_config,
    qa_pipeline_config=QAPipelineConfig(
        preprocessor_config=QueryPreprocessorConfig(
            denoising_config=None, # None | by default
            enhancing_config=None, # None | by default
            decomposition_config=None, # None | by default
        ),
        reasoner_config=KnowledgeGraphReasonerConfig(
            reasoner_name='weak', # 'weak' | 'medium'
            reasoner_hyperparameters=WeakKGReasonerConfig() # WeakKGReasonerConfig() |  MediumKGReasonerConfig()
        )
    )
)

In [40]:
personalai = PersonalAI(pai_config)

No sentence-transformers model found with name ../../../models/intfloat/multilingual-e5-small. Creating a new one with mean pooling.


In [26]:
personalai.mem_pipeline.clear_kv_caches()
personalai.qa_pipeline.clear_kv_caches()

In [41]:
print("mem kv_cache info:")
pprint(personalai.mem_pipeline.get_cache_stat())
print("mem agent tgen info:")
pprint(personalai.mem_pipeline.get_agent_tgen_stat())
print("qa kv_cache info:")
pprint(personalai.qa_pipeline.get_cache_stat())
print("qa agent tgen info:")
pprint(personalai.qa_pipeline.get_agent_tgen_stat())

mem kv_cache info:
{'extractor': {'thesises_extraction_solver': 4,
               'triplets_extraction_solver': 4},
 'updator': {'replace_hyper_solver': 0, 'replace_simple_solver': 0}}
mem agent tgen info:
{'extractor': {'thesises_extraction_solver': {'generated_tokens_amount': {'count': 12,
                                                                          'count_not_null': 12,
                                                                          'max': 103,
                                                                          'mean': 65.5,
                                                                          'median': 63.0,
                                                                          'min': 33,
                                                                          'std': 24.875,
                                                                          'sum': 786},
                                              'inference_elapsed_time': {'count': 12,


In [28]:
personalai.kg_model.clear()

In [29]:
personalai.kg_model.count_items(detailed=True)

{'graph_info': {'triplets': {'simple': 0, 'hyper': 0, 'episodic': 0},
  'nodes': {'object': 0, 'hyper': 0, 'episodic': 0}},
 'embeddings_info': {'nodes': {'object': {'nodes_dense': 0,
    'nodes_sparse_bm25': 0},
   'hyper': {'nodes_dense': 0, 'nodes_sparse_bm25': 0},
   'episodic': {'nodes_dense': 0, 'nodes_sparse_bm25': 0}},
  'triplets': {'triplets_dense': 0, 'triplets_sparse_bm25': 0}},
 'nodestree_info': None}

In [30]:
TEXT_EXAMPLES = [
    "Sasha was walking along the highway.", 
    "Masha was walking along the highway.", 
    "The ship was sailing along the water canal.", 
    "The motorboat was sailing along the river."]

In [31]:
extracted_triplets = []
for i, example in enumerate(TEXT_EXAMPLES):
    print(f"{i+1}. {example}")
    tmp_extracted_triplets, _ = personalai.update_memory(example)
    extracted_triplets += tmp_extracted_triplets
    print("extracted triplets: ", len(tmp_extracted_triplets))

1. Sasha was walking along the highway.
extracted triplets:  8
2. Masha was walking along the highway.
extracted triplets:  10
3. The ship was sailing along the water canal.
extracted triplets:  8
4. The motorboat was sailing along the river.
extracted triplets:  6


In [32]:
pprint(personalai.kg_model.count_items(detailed=True))
personalai.kg_model.check_consistency()

{'embeddings_info': {'nodes': {'episodic': {'nodes_dense': 4,
                                            'nodes_sparse_bm25': 4},
                               'hyper': {'nodes_dense': 6,
                                         'nodes_sparse_bm25': 6},
                               'object': {'nodes_dense': 9,
                                          'nodes_sparse_bm25': 9}},
                     'triplets': {'triplets_dense': 15,
                                  'triplets_sparse_bm25': 15}},
 'graph_info': {'nodes': {'episodic': 4, 'hyper': 6, 'object': 9},
                'triplets': {'episodic': 16, 'hyper': 11, 'simple': 5}},
 'nodestree_info': None}


True

In [33]:
print("mem kv_cache info:")
pprint(personalai.mem_pipeline.get_cache_stat())
print("mem agent tgen info:")
pprint(personalai.mem_pipeline.get_agent_tgen_stat())

mem kv_cache info:
{'extractor': {'thesises_extraction_solver': 4,
               'triplets_extraction_solver': 4},
 'updator': {'replace_hyper_solver': 0, 'replace_simple_solver': 0}}
mem agent tgen info:
{'extractor': {'thesises_extraction_solver': {'generated_tokens_amount': {'count': 12,
                                                                          'count_not_null': 12,
                                                                          'max': 103,
                                                                          'mean': 65.5,
                                                                          'median': 63.0,
                                                                          'min': 33,
                                                                          'std': 24.875,
                                                                          'sum': 786},
                                              'inference_elapsed_time': {'count': 12,


In [34]:
answer, rinfo = personalai.answer_question("Did Masha walk along the highway?")
print(answer)

Yes


In [35]:
answer, rinfo = personalai.answer_question("Did Katya walk along the highway?")
print(answer)

<|NotEnoughtInfo|>


In [36]:
answer, rinfo = personalai.answer_question("Did ship was sailing along the water canal?")
print(answer)

Yes


In [37]:
print("qa kv_cache info:")
pprint(personalai.qa_pipeline.get_cache_stat())
print("qa agent tgen info:")
pprint(personalai.qa_pipeline.get_agent_tgen_stat())

qa kv_cache info:
{'QAPipeline': 3,
 'answers_aggregator': {'AnswersAggregator': 3,
                        'subanswers_summarisation_solver': 0},
 'kg_reasoner': {'KnowledgeGraphReasoner': 3,
                 'reasoner': {'WeakKGReasoner': 3,
                              'answer_generator': {'QALLMGenerator': 3,
                                                   'answer_generator_solver': 3},
                              'knowledge_comparator': {'KnowledgeComparator': 3},
                              'knowledge_retriever': {'KnowledgeRetriever': 3,
                                                      'triplets_filter': {'TripletsFilter': 3},
                                                      'triplets_retriever': {'BeamSearchTripletsRetriever': 6}},
                              'query_parser': {'QueryLLMParser': 3,
                                               'kw_extraction_solver': 3}}},
 'query_preprocessor': {'QueryPreprocessor': 3,
                        'decomposer': N

In [38]:
personalai.kg_model.count_items()

{'graph_info': {'triplets': 32, 'nodes': 19},
 'embeddings_info': {'nodes': 19, 'triplets': 15},
 'nodestree_info': None}

In [42]:
del personalai